# Quickstart: anomaly and falsification detection on greenhouse IoT data

This notebook runs the project pipeline step by step. It works locally and on Kaggle.
Every function lives in `src/agri_anomaly/`; the notebook only calls it and shows the results.

**On Kaggle**: clone the repository first (next cell), add the Zenodo dataset as an input, and set `RAW_FILE`.\n\nLocally, the notebook uses `data/raw/Greenhouse-1.xlsx` (Zenodo 5793685). The first read of the .xlsx takes about a minute; later runs use a cached copy.

In [ ]:
# --- Kaggle only: uncomment ---
# !git clone https://github.com/<you>/agri-iot-anomaly.git
# %cd agri-iot-anomaly
# !pip install -q -r requirements.txt

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Image, Markdown, display

from agri_anomaly.config import load_config
from agri_anomaly import pipeline
from agri_anomaly.utils import get_logger

get_logger()
cfg = load_config(ROOT / "config.yaml")

# Real data (default): data/raw/Greenhouse-1.xlsx, as set in config.yaml.
# On Kaggle, set RAW_FILE to the input path, e.g. "/kaggle/input/<name>/Greenhouse-1.xlsx".
# SYNTHETIC = True runs on generated data instead (no dataset needed).
RAW_FILE = None
SYNTHETIC = False
if RAW_FILE:
    cfg["paths"]["raw_file"] = RAW_FILE

## 1. Data: load, audit, clean, resample, split

In [ ]:
meta = pipeline.prepare_data(cfg, synthetic=SYNTHETIC)
meta

In [ ]:
display(Markdown((ROOT / "reports/data_audit.md").read_text()[:4000]))

In [ ]:
train = pd.read_parquet(ROOT / "data/processed/train.parquet")
s = train["sensor_id"].iloc[0]
train[train["sensor_id"] == s].set_index("timestamp")[cfg["data"]["variables"]].plot(subplots=True, figsize=(14, 5), title=s);

## 2. Simulation: inject faults and attacks (ground truth in `reports/attack_log.csv`)

In [ ]:
events = pipeline.simulate_attacks(cfg)
events.groupby(["split", "category", "scenario"]).size().unstack(0)

## 3. Features

In [ ]:
names = pipeline.build_features(cfg)
pd.read_parquet(ROOT / "data/features/test_attacked.parquet")[names].describe().T.round(3)

## 4. Training (stage 1 detectors and stage 2 classifiers, thresholds on validation)

In [ ]:
summary = pipeline.train_models(cfg)
summary

## 5. Evaluation on the test split

In [ ]:
res = pipeline.evaluate_models(cfg)
res["stage1"].round(3)

In [ ]:
res["per_scenario"].pivot(index="model", columns="scenario", values="event_recall").round(2)

In [ ]:
res["end_to_end"].round(3)

In [ ]:
for f in ["roc_pr_curves.png", "event_recall_per_scenario.png", "end_to_end_cm_best_unsupervised.png"]:
    display(Image(filename=str(ROOT / "reports/figures" / f)))

In [ ]:
for f in sorted((ROOT / "reports/figures").glob("timeline_*.png")):
    display(Image(filename=str(f)))

## Next
* Run `streamlit run app/streamlit_app.py` on your computer for the dashboard.
* Change parameters in `config.yaml` (scenarios, windows, models) and rerun from the stage you changed.